# HF → Kaggle dataset bridge (bundle-y1)

AutoDL → Kaggle direct upload is blocked by IP routing (Kaggle's GCS resumable-upload endpoint lives in an IP range unreachable from AutoDL CN POPs). This notebook pulls the bundle from HuggingFace and creates the Kaggle dataset from inside Kaggle's own network.

**Before Run All:**
1. Add-ons → Settings → **Internet: On**
2. Add-ons → Secrets → add secret named `HF_TOKEN` with your `hf_…` token, and **attach it to this notebook**. Required — the HF repo `birdclef-2026-ckpts` is private.
3. Run All

After it finishes, the dataset lives at `tiantanghuaxiao/birdclef-2026-bundle-y1` and you can attach it to `submit_v5.ipynb`.

In [ ]:
# 1. Install latest huggingface_hub if needed
!pip install -q -U huggingface_hub>=0.28

In [ ]:
# 2. Download bundle subfolder from HF (private repo — needs HF_TOKEN from Kaggle Secrets)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
import os, shutil

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'HF_TOKEN secret missing or malformed'

STAGING = '/kaggle/working/staging'
shutil.rmtree(STAGING, ignore_errors=True)
os.makedirs(STAGING, exist_ok=True)

snapshot_download(
    repo_id='Tiantanghuaxiao/birdclef-2026-ckpts',
    repo_type='model',
    allow_patterns=['bundle_y1/*'],
    local_dir=STAGING,
    token=HF_TOKEN,
)

BUNDLE = os.path.join(STAGING, 'bundle_y1')
print('downloaded files:')
!du -sh {BUNDLE}/* {BUNDLE}/wheels/* 2>/dev/null

In [ ]:
# 3. Ensure dataset-metadata.json has the intended slug
import json, pathlib
meta_path = pathlib.Path(BUNDLE) / 'dataset-metadata.json'
meta = json.loads(meta_path.read_text())
# Adjust if you want a different slug:
meta['id'] = 'tiantanghuaxiao/birdclef-2026-bundle-y1'
meta['title'] = 'birdclef-2026-bundle-y1'
meta.setdefault('licenses', [{'name': 'CC0-1.0'}])
meta_path.write_text(json.dumps(meta, indent=2))
print(meta)

In [ ]:
# 4. Kaggle CLI is pre-authed inside notebooks (via ~/.kaggle/kaggle.json that
# Kaggle auto-mounts for notebook owners). Just verify the CLI talks to the API:
!kaggle --version
!kaggle datasets list --user tiantanghuaxiao | head -3

In [ ]:
# 5. Create the dataset on Kaggle from within Kaggle's network
#    --dir-mode zip packs wheels/ into wheels.zip (Kaggle auto-unzips at mount time)
!cd {BUNDLE} && kaggle datasets create -p . --dir-mode zip

In [ ]:
# 6. Verify
!kaggle datasets list --user tiantanghuaxiao | head